# 12 - Interactive compared with non-interactive (take-home)

**ICSSI 2026, Using LLMs via the API**

There are two ways to run the model. In the interactive mode a human stays in the loop,
reading each reply and deciding the next message, which suits chat and exploration. In the
non-interactive mode the model runs unattended, working through a task on its own, which
suits pipelines and agents. We show one of each: a multi-turn `Conversation`, and an
`Agent` that loops over rounds by itself.

For unattended bulk jobs over many items, the Batch API is a third option at half price,
and it is covered in notebook 01.

In [ ]:
# fetch the repo files (claude_kit.py and tutorial_data/) when running in Colab
import os, sys, subprocess
REPO_URL = "https://github.com/LarremoreLab/icssi-2026-hackathon.git"  # update after you push to GitHub

if "google.colab" in sys.modules and not os.path.exists("claude_kit.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))

# this notebook lives in take_home/; walk up to the repo root so claude_kit.py,
# .env and tutorial_data/ all resolve no matter which folder the kernel started in
while not os.path.exists("claude_kit.py") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("working directory:", os.getcwd())
assert os.path.exists("claude_kit.py"), (
    "claude_kit.py was not found. Run this notebook from inside the cloned repo folder, "
    "or set REPO_URL above and run this cell again in Colab."
)

In [ ]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv
    except ImportError:
        pip_install("python-dotenv")
        from dotenv import load_dotenv
    load_dotenv(override=True)

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")
MODEL = "claude-haiku-4-5"

## Interactive: a multi-turn conversation

A conversation where you read each reply and decide the next message. The `Conversation`
helper remembers the history for you, since the API itself is stateless. Here we simulate
two turns. In a real application the second turn would come from the user.

In [ ]:
from claude_kit import ClaudeClient, Conversation, CostTracker, Agent

tracker = CostTracker(budget_usd=0.75)
kit = ClaudeClient(tracker=tracker, max_tokens=400)

chat = Conversation(kit, system="You are a helpful science-of-science research assistant.")
print(chat.send("I have a dataset of grant abstracts. Suggest two research questions."))
print("\n---\n")
print(chat.send("Pick the more feasible one for a two-day hackathon and say why."))

## Non-interactive: an agent that loops over rounds

An agent is just a loop that runs without a human between steps. Each trip through the loop
is a round. In a round Claude thinks, optionally calls a tool such as web search, sees the
result, and decides whether it is done. When a server-side tool loop hits its internal cap,
the response comes back with `stop_reason` equal to `pause_turn`, and the loop re-sends to
let it continue, which is the next round.

The `Agent` class in `claude_kit` wraps this loop, caps the number of rounds, and logs the
cost of each round, including the per-search web charge.

In [ ]:
agent = Agent(
    kit,
    system="You are a meticulous research assistant for science-of-science scholars. "
           "Be concise and always cite source URLs.",
    max_rounds=5,
)

result = agent.run(
    "Find two recent papers, from 2023 or later, about citation dynamics or 'sleeping "
    "beauties' in science. For each, give the title and a two-sentence summary with a link."
)
print("\n=== ANSWER ===\n")
print(result["answer"])
print("\nrounds used:", result["rounds"], "| stop_reason:", result["stop_reason"])
print()
tracker.report()

## Choosing a mode

Choose the interactive mode when a human steers, such as demos, exploration, and ambiguous
tasks. Choose an unattended agent when the task is well specified and benefits from the
model taking several steps on its own. Choose the Batch API, from notebook 01, when you
have many independent items and do not need answers right now, since it is half price for
the same work.